# World Bank Pink Sheet — monthly Colombo tea auction price

The World Bank's "Pink Sheet" is a monthly commodity price table published
since 1960. CeyNex uses exactly one column from it: **`Tea, Colombo`** — the
Colombo auction price, in USD per kilogram.

Connector: `ceynex/data/connectors/pinksheet.py` (owner: M1 Dinapura).

This is the **cleanest series in the whole project**: monthly, machine-readable,
no gaps, and maintained by an institution that will still be publishing it next
year. Every other price source CeyNex has is annual, patchy, or both.

In [1]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

import _common as cx

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Is the workbook staged?

In [2]:
AGRI = cx.agriculture_raw_dir()
print("agriculture raw dir:", AGRI or "none found")
print()

workbook = cx.status(
    "Pink Sheet",
    (AGRI / "pinksheet") if AGRI else None,
    "CMO-Historical-Data-Monthly.xlsx",
    how="free download from worldbank.org — search 'Pink Sheet monthly historical'.",
)

agriculture raw dir: /ml/CeyNex/ceynex-core/data/raw

Pink Sheet
  staged: NO — the analysis cells below will skip.
  expected at: /ml/CeyNex/ceynex-core/data/raw/pinksheet/CMO-Historical-Data-Monthly.xlsx
  how to get it: free download from worldbank.org — search 'Pink Sheet monthly historical'.


## 2. The header is not on a fixed row

The `Monthly Prices` sheet has several title and unit rows before the real
header, and the count has changed between releases. Hard-coding `header=4`
works until the day the World Bank adds a line.

The connector instead **searches for the row containing `Tea, Colombo`** and
treats that as the header. It is a small thing that stops a routine upstream
edit from silently shifting every column by one.

Rows are then kept only if the first column matches `YYYYMdd` — e.g. `1960M01`.
That drops the footnote rows at the bottom of the sheet without needing to know
how many there are.

## 3. The missing-value marker is a character, not a blank

Missing observations are written as the ellipsis character `…`. Two
consequences:

- The column arrives as `object` dtype, not numeric. Any arithmetic on it fails
  or silently produces nonsense.
- The file is sometimes served mis-encoded, turning `…` into `â€¦`.

The connector coerces every price column with `pd.to_numeric(errors="coerce")`
and replaces both spellings, so Arrow can write one stable Parquet schema.
Without this, the Parquet write fails on a mixed-type column — a confusing
error a long way from its cause.

## 4. Load

In [3]:
prices = None
if workbook is not None:
    sheet = pd.read_excel(workbook, sheet_name="Monthly Prices", header=None)
    header_row = next(
        i for i, row in sheet.iterrows() if "Tea, Colombo" in row.astype(str).tolist()
    )
    data = sheet.iloc[header_row + 1:].copy()
    data.columns = sheet.iloc[header_row].astype(str).str.strip()
    data = data.rename(columns={data.columns[0]: "period"}).dropna(subset=["period"])
    data = data[data["period"].astype(str).str.match(r"^\d{4}M\d{2}$")].copy()

    price_columns = data.columns.drop("period")
    data[price_columns] = data[price_columns].apply(pd.to_numeric, errors="coerce")
    data = data.replace({"…": pd.NA, "â€¦": pd.NA})

    prices = data[["period", "Tea, Colombo"]].copy()
    prices["date"] = pd.to_datetime(
        prices["period"].str.replace("M", "-", regex=False) + "-01", format="%Y-%m-%d"
    )
    prices["price_usd_per_kg"] = pd.to_numeric(prices["Tea, Colombo"], errors="coerce")

    print(f"header found on sheet row {header_row}")
    print(f"{len(data):,} monthly rows, {data['period'].min()} to {data['period'].max()}")
    print(f"{prices['price_usd_per_kg'].notna().sum():,} months with a tea price")
    print(f"{len(price_columns)} commodity columns in the sheet; CeyNex uses 1")
    display(prices.tail())
else:
    print("skipped — no workbook staged")

skipped — no workbook staged


## 5. The tea price since 1960

In [4]:
if prices is not None:
    series = prices.dropna(subset=["price_usd_per_kg"]).set_index("date")["price_usd_per_kg"]
    ax = series.plot(title="Tea, Colombo auction — nominal USD per kg")
    ax.set_ylabel("USD / kg")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()

    print(f"range   : {series.index.min():%Y-%m} to {series.index.max():%Y-%m}")
    print(f"min     : {series.min():.2f} USD/kg  ({series.idxmin():%Y-%m})")
    print(f"max     : {series.max():.2f} USD/kg  ({series.idxmax():%Y-%m})")
    print(f"latest  : {series.iloc[-1]:.2f} USD/kg  ({series.index[-1]:%Y-%m})")
else:
    print("skipped — no data")

skipped — no data


> **These are nominal prices, not inflation-adjusted.** A 1970s peak on this
> chart is not comparable to a 2020s one in real terms. The Pink Sheet does
> publish a deflated table on a separate sheet; CeyNex does not currently use
> it. Say "nominal" out loud if you show this chart.

## 6. Gap check

The claim that this series has no gaps should be tested, not repeated.

In [5]:
if prices is not None:
    series = prices.dropna(subset=["price_usd_per_kg"]).set_index("date")["price_usd_per_kg"]
    expected = pd.date_range(series.index.min(), series.index.max(), freq="MS")
    missing = expected.difference(series.index)
    print(f"expected months : {len(expected):,}")
    print(f"present         : {len(series):,}")
    print(f"missing         : {len(missing)}")
    if len(missing):
        print("  first few:", [f"{d:%Y-%m}" for d in missing[:12]])
else:
    print("skipped — no data")

skipped — no data


## 7. Annual average, for joining to the annual sources

Everything else in CeyNex is annual. `DataCleaner`'s rule for prices is to
**average** within the period — prices are a rate, so averaging is right;
volumes are a quantity, so they are summed instead.

In [6]:
if prices is not None:
    series = prices.dropna(subset=["price_usd_per_kg"]).set_index("date")["price_usd_per_kg"]
    annual = series.resample("YE").agg(["mean", "min", "max", "count"])
    annual.index = annual.index.year
    annual = annual[annual["count"] == 12]
    display(annual.tail(15).round(2))
    print("Years with fewer than 12 months are excluded — a partial year's mean")
    print("is not comparable to a full year's.")
else:
    print("skipped — no data")

skipped — no data
